# Demo: Data Access with Provider Gateway

This notebook demonstrates the new approach to data access using the `provider` gateway. With a single initialization, you can access experiment data as DataFrames for analysis.

**Key Features:**

1. **Database Location:** Place your DuckDB database file (e.g., `DB.duckdb`) in the `data/` directory. The provider automatically looks for the database there during initialization. 
2. **Schema Inspection:** Use `provider.schema()` to display the table structure, including all available columns and their types.
3. **Extending with Custom Queries:** You can add new queries to the loader and make them accessible via provider, allowing for custom data access patterns as your analysis needs grow. 
4. **Safe Connection Handling:** The provider manages the database connection for you. If you need to close the connection manually (e.g., at the end of a script), call `provider.close()`. 

- Initialization is one line: `provider.init()`
- Access any table or query as a DataFrame: 
  `provider.df()`, `provider.plates()`, `provider.slots()`, `provider.signals()`, `provider.data_origin()`, `provider.plate_slots()`, `provider.grouped_data()`
- The `src/loader.py` contains the actual queries and logic


In [ ]:
# 1. Environment and Provider Initialization
%load_ext autoreload
%autoreload 2
%run ../config/preamble.py

from src import provider

# Initialize the provider gateway
provider.init()

##  Data exploration 

After initialization, you can first explore the database 

In [ ]:
# Show the schema of the current table
provider.schema()

In [ ]:
# List all available plates
provider.plates()

In [ ]:
# List all slots for a specific plate (replace 'Plate_1' with a real plate name)
provider.slots(22)

In [ ]:
# List all (plate, slot) pairs
provider.plate_slots()

In [ ]:
# List all signals for a plate and slot (replace with real values)
signals = provider.signals(22, 1)
signals.head(15)

In [ ]:
# List all data origins for a plate and slot (replace with real values)
provider.data_origin(22, 1)

## DataFrames
You can load a subset of data as a DataFrame using provider.df()\
Use these options to flexibly select, filter, and limit the data you load as a DataFrame for your analysis.

* plate and slot (required),
* columns to select,
* signal names,
* data origin,
* WCS_Y_mm value range,
* row limit,
* and whether to order by time.

In [ ]:
# 1. Get all data for a specific plate and slot
df = provider.df(22, 11)
df.sample(15)


In [ ]:
# 2. Get all current measurements (in amps) for plate 22, slot 1, only from HF_Data
df = provider.df(22, 1, signals=["CURRENT|4"], data_origin="HF_Data", fields=["Time", "Value", "Unit"])
df.head(10)

In [ ]:
# 3. Get all contour deviation values 
df = provider.df(22, 1, signals=["CONT_DEV|6"], fields=["Time", "Value", "Description"])

df = df[df["Value"] > 0]
if not df.empty:
    display(df.sample(15))
else:
    print("No rows with Value > 0.")

In [ ]:
# 4. Get all WCSPosition values for plate 22, slot 1, in a specific time range
df = provider.df(24, 1, signals=["WCSPosition"], fields=["Time", "Value"])
df.sample(20)

In [ ]:
# 5. Get all data for plate 22, slot 1, where Axis is 'A'
df = provider.df(22, 1, fields=["Time", "Signal", "Value", "Axis"])
df = df[df["Axis"] == "A"]
df.head(15)

In [ ]:
# 6. Get the first 50 rows of all signals for plate 22, slot 1, ordered by time
df = provider.df(22, 1, limit=50, order_by_time=True)
df.head(15)

## GROUP_DATA

You can use provider.group_data() to group by one or more columns (such as Signal, DataOrigin, or Axis) and apply aggregation functions: mean, sum, count, min, or max to selected columns.

In [ ]:
# 1. Mean value per signal for a plate and slot
df = provider.group_data(group_by=["Signal"], agg={"Value": "mean"}, plate=22, slot=1)
df.head()

In [ ]:
# 2. Maximum value per DataOrigin
df = provider.group_data(group_by=["DataOrigin"], agg={"Value": "max"}, plate=22, slot=1)
df.head()

In [ ]:
# 3. Count of records per Axis
df = provider.group_data(group_by=["Axis"], agg={"Value": "count"}, plate=22, slot=1)
df.head()

In [ ]:
# 4. Sum of Value per Signal and DataOrigin
df = provider.group_data(group_by=["Signal", "DataOrigin"], agg={"Value": "sum"}, plate=22, slot=1)
df.head()

## Provider in other scripts

The provider module can also be used in your data processing scripts or modules, not just in notebooks. For example, you can import provider in your data_processing.py and use provider.df() or provider.group_data() to load and process data in the same way as in your notebook.

In [ ]:
# Import  the module 
from src import provider

# 1. Environment and Provider Initialization
%load_ext autoreload
%autoreload 2
%run ../config/preamble.py
provider.init()

# Now you can process df as needed in your scripts
selected_df = provider.df(22, 1, fields=["Time", "Value"])
